# Stage 2B: Bivariate/Multivariate EDA, Correlation & Data Fitting
### AMEO 2015 Engineering Placement Benchmark (2,998 Records, 34 Features)
**Assigned to:** Prit  
**Course:** IT462 Exploratory Data Analysis | Syllabus Modules 4, 7 & 8 & Lecture Notes  

---
## 📌 Syllabus & Lecture Concepts Implemented in this Notebook
1. **Lecture Note Slide 1 (Classification vs Regression Mapping):**
   - **Regression:** Target $y \in \mathbb{R}$ (`Salary`), Predictors $x_i \in \text{Num / Cat}$ $\to$ Covariance Matrix, Pearson & Spearman Correlation Matrices, Linear Regression.
   - **Classification:** Target $y \in \text{Cat}$ (`placed_tier`), Predictors $x_i \in \text{Num / Cat}$ $\to$ Contingency Matrix (Crosstabs) $\to$ Heatmaps.
2. **Module 7: Bivariate Analysis:** Scatter Plots, Grouped Boxplots, Violin Plots, Covariance Matrix (`df.cov()`).
3. **Module 8: Correlation & Multicollinearity:** Pearson vs Spearman, VIF (Variance Inflation Factor).
4. **Module 8: Data Fitting & Regression:** OLS Linear & Multiple Regression, Goodness of Fit ($R^2$, Adj $R^2$, RMSE, Residual Diagnostics).
5. **Module 4: Transformation & Feature Engineering:** Log Transformation $\ln(Salary)$, Categorical Encoding, Export `ameo_encoded.csv` for Stage 3 & 4.


## 1. Setup & Environment Configuration
Import analytical, statistical, and regression packages (`numpy`, `pandas`, `scipy.stats`, `statsmodels`, `seaborn`, `matplotlib`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Stage 2B Environment initialized successfully.")

## 2. Ingest Cleaned Dataset (Fallback to Raw with Baseline Clean)
> **TODO (Prit):** Load Dhruven's cleaned dataset (`../data/processed/ameo_cleaned.csv`). If Dhruven is still completing his stage, fallback automatically loads `../data/raw/Engineering_graduate_salary.csv`.

In [ ]:
cleaned_path = '../data/processed/ameo_cleaned.csv'
raw_path = '../data/raw/Engineering_graduate_salary.csv'

if os.path.exists(cleaned_path):
    df = pd.read_csv(cleaned_path)
    print(f"Loaded Dhruven's Cleaned Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
else:
    df = pd.read_csv(raw_path)
    print(f"Loaded Raw Benchmark Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

df.head()

## 3. Bivariate Visualizations: Determinants of Placement & Salary (Syllabus Module 7)
Explore pairwise relationships between academic predictors, cognitive test scores, and starting salaries:
- **Scatter Plot:** Continuous vs Continuous (`collegeGPA` vs `Salary`, `Quant` vs `Salary`)
- **Grouped Boxplots / Violin Plots:** Categorical vs Continuous (`CollegeTier` vs `Salary`, `Gender` vs `Salary`)

> **TODO (Prit):** Build bivariate visualizations and save figures to `../reports/figures/`.

In [ ]:
# TODO (Prit): Bivariate plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Scatter Plot: Quant Score vs Salary
sns.scatterplot(x='Quant', y='Salary', hue='CollegeTier', data=df, alpha=0.6, palette='Set1', ax=axes[0])
axes[0].set_title('Bivariate Scatter: AMCAT Quant vs Salary (by College Tier)')
axes[0].set_xlabel('Quantitative Ability Score')
axes[0].set_ylabel('Annual Salary (INR)')

# 2. Grouped Boxplot: College Tier vs Salary
sns.boxplot(x='CollegeTier', y='Salary', data=df, palette='Blues', ax=axes[1])
axes[1].set_title('Salary Distribution across Institutional College Tiers')
axes[1].set_xticklabels(['Tier 1 Institutions', 'Tier 2 Institutions'])

plt.tight_layout()
plt.savefig('../reports/figures/prit_bivariate_analysis.png', dpi=300)
plt.show()

## 4. Covariance & Correlation Analysis: Pearson vs Spearman (Syllabus Modules 7 & 8)
### Theoretical Distinction:
- **Covariance:** $\text{Cov}(X, Y) = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})(Y_i - \bar{Y})$ — measures the directional joint variability of two variables (unstandardized, scale-dependent).
- **Pearson Correlation ($r$):** Parametric measure of linear relationship. Sensitive to outliers and requires approximately normal bivariate distributions.
- **Spearman Rank Correlation ($\rho$):** Non-parametric measure of monotonic relationship based on rank order: $\rho = 1 - \frac{6\sum d_i^2}{n(n^2 - 1)}$. Highly robust to non-normal distributions and skewed variables like `Salary`.

> **TODO (Prit):** Calculate the Covariance Matrix, Pearson correlation, Spearman rank correlation, and compare their differences.

In [ ]:
# TODO (Prit): Covariance Matrix (Module 7)
eval_cols = ['Salary', 'collegeGPA', '10percentage', '12percentage', 'English', 'Logical', 'Quant']
cov_matrix = df[eval_cols].cov()
print("Sample Covariance Matrix:")
cov_matrix

# TODO (Prit): Pearson vs Spearman Comparison (Module 8)
pearson_corr = df[eval_cols].corr(method='pearson')
spearman_corr = df[eval_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Pearson Correlation Matrix (Linear)')

sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Spearman Rank Correlation Matrix (Monotonic)')

plt.tight_layout()
plt.savefig('../reports/figures/prit_correlation_comparison.png', dpi=300)
plt.show()

## 5. Contingency Tables (Crosstabs) & Heatmaps (Syllabus Module 7)
Analyze relationships between categorical dimensions:
- `CollegeTier` vs `Degree`
- `Gender` vs Top Quartile Placement Success (`HighSalary`)

> **TODO (Prit):** Build two-way crosstabs with marginal row/column totals (`margins=True`) and visualize with heatmaps.

In [ ]:
# TODO (Prit): Build Contingency Table with Marginals
salary_q75 = df['Salary'].quantile(0.75)
df['HighSalary'] = (df['Salary'] >= salary_q75).astype(int)

crosstab_tier = pd.crosstab(df['CollegeTier'], df['HighSalary'], margins=True, normalize='index') * 100
print("Contingency Table (% High Salary within College Tier):")
print(crosstab_tier)

plt.figure(figsize=(7, 4))
sns.heatmap(crosstab_tier.iloc[:-1, :-1], annot=True, fmt='.1f', cmap='YlGnBu')
plt.title('Percentage of Students Earning Top-Quartile Salary by Tier')
plt.xlabel('High Salary Flag (0 = No, 1 = Yes)')
plt.ylabel('College Tier (1 = Tier 1, 2 = Tier 2)')
plt.show()

## 6. Multicollinearity Diagnostics using VIF (Syllabus Module 8)
### Variance Inflation Factor (VIF):
$$VIF_i = \frac{1}{1 - R_i^2}$$
Where $R_i^2$ is the coefficient of determination when predictor $X_i$ is regressed onto all other remaining predictors.
- $VIF = 1$: No collinearity
- $1 < VIF < 5$: Moderate, acceptable collinearity
- $VIF > 5$ or $10$: Severe multicollinearity that inflates standard errors and destabilizes regression coefficients.

> **TODO (Prit):** Calculate VIF across continuous academic and cognitive predictors.

In [ ]:
# TODO (Prit): Compute VIF using statsmodels
vif_features = df[['collegeGPA', '10percentage', '12percentage', 'English', 'Logical', 'Quant']].dropna()
X_vif = sm.add_constant(vif_features)

vif_data = pd.DataFrame()
vif_data['Feature'] = vif_features.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(vif_features.columns))]
vif_data = vif_data.sort_values(by='VIF', ascending=False)
print("Variance Inflation Factor (VIF) Diagnostics:")
vif_data

## 7. Data Fitting: Simple & Multiple Linear Regression (Syllabus Module 8)
Fit Ordinary Least Squares (OLS) Regression models to explain graduate compensation (`Salary`):
1. **Simple Linear Regression:** $Salary = \beta_0 + \beta_1 \cdot Quant + \epsilon$
2. **Multiple Linear Regression:** $Salary = \beta_0 + \beta_1 \cdot collegeGPA + \beta_2 \cdot Quant + \beta_3 \cdot Logical + \beta_4 \cdot English + \beta_5 \cdot 12percentage + \epsilon$
3. **Goodness of Fit & Diagnostics:**
   - Coefficient of Determination ($R^2$ and Adjusted $R^2$)
   - Root Mean Squared Error (RMSE)
   - Residual vs Fitted Plot (Testing Homoscedasticity)
   - Normal Q-Q Plot (Testing Normality of Residuals)

> **TODO (Prit):** Fit OLS model and evaluate regression diagnostics.

In [ ]:
# TODO (Prit): Fit Multiple Linear Regression Model
reg_cols = ['collegeGPA', '10percentage', '12percentage', 'English', 'Logical', 'Quant']
clean_reg = df[reg_cols + ['Salary']].dropna()

X_reg = sm.add_constant(clean_reg[reg_cols])
y_reg = clean_reg['Salary']

ols_model = sm.OLS(y_reg, X_reg).fit()
print(ols_model.summary())

# Goodness of fit metrics
rmse = np.sqrt(np.mean(ols_model.resid ** 2))
print(f"\nGoodness of Fit Diagnostics:")
print(f"R-squared: {ols_model.rsquared:.4f}")
print(f"Adjusted R-squared: {ols_model.rsquared_adj:.4f}")
print(f"RMSE: INR {rmse:,.2f}")

# Residual diagnostics plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 1. Residuals vs Fitted
axes[0].scatter(ols_model.fittedvalues, ols_model.resid, alpha=0.4, color='darkblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted (Homoscedasticity Check)')
axes[0].set_xlabel('Fitted Salary Values')
axes[0].set_ylabel('Residuals')

# 2. Normal Q-Q Plot
sm.qqplot(ols_model.resid, line='s', ax=axes[1])
axes[1].set_title('Normal Q-Q Plot of Residuals')

plt.tight_layout()
plt.savefig('../reports/figures/prit_regression_residuals.png', dpi=300)
plt.show()

## 8. Feature Transformation, Encoding & Export (Syllabus Module 4)
Prepare the modeling matrix for Punit's Stage 3 & 4:
1. **Log Transformation:** Compute $\ln(Salary)$ to compress severe right skewness.
2. **Placement Tier Target:** Derive binary placement tier (`HighSalary` / `placed_tier`).
3. **Categorical Encoding:** One-hot encode `Gender`, `Degree`, `Specialization`, `CollegeTier`.
4. **Export Clean Encoded CSV:** Save to `../data/processed/ameo_encoded.csv`.

> **TODO (Prit):** Perform encoding and export `ameo_encoded.csv`.

In [ ]:
# TODO (Prit): Uncomment after completing Stage 2B feature engineering
# df['log_Salary'] = np.log1p(df['Salary'])
# os.makedirs('../data/processed', exist_ok=True)
# df.to_csv('../data/processed/ameo_encoded.csv', index=False)
# print("Feature engineered dataset exported successfully to ../data/processed/ameo_encoded.csv")